# CS4120 Final — Part 2, Evaluation

Apples-to-apples comparison of three models on the **same test split** (216 records, 0 combat leakage from train):

| Model | Where it comes from | Strategy |
|---|---|---|
| **Template** | `template_generate` below | Deterministic slot-filling from the JSON — our no-ML floor |
| **N-gram (KN-3)** | `models/ngram/predictions_test.jsonl` | Trigram Kneser-Ney LM conditioned on tag prefix |
| **T5-small** | `models/t5_small/predictions_test.jsonl` | Fine-tuned seq2seq on `linearize_for_t5` → narration |

Metrics:
- **BLEU-4** (sacrebleu) — corpus-level
- **ROUGE-L** — per-sample, averaged
- **BERTScore-F1** — semantic similarity (optional; heavy)
- **Entity faithfulness** — fraction of generations that mention the gold actor name and at least one target name (our custom faithfulness metric)
- **Length ratio** — avg words(pred) / words(gold) — diagnostic for over/under-generation

In [1]:
# sacrebleu/rouge_score installed in local venv; bert-score skipped


In [2]:
import sys, json, math
from pathlib import Path
from collections import defaultdict
import sacrebleu
from rouge_score import rouge_scorer

sys.path.insert(0, '/Users/xuanhu/Library/CloudStorage/OneDrive-Personal/20260107-20260419/CS4120/dnd_project/part2')
from fireball_preprocess import linearize_for_t5

DATA_DIR = Path('/Users/xuanhu/Library/CloudStorage/OneDrive-Personal/20260107-20260419/CS4120/dnd_project/part2' + '/processed')
NGRAM_PRED = Path('/Users/xuanhu/Library/CloudStorage/OneDrive-Personal/20260107-20260419/CS4120/dnd_project/part2' + '/models/ngram/predictions_test.jsonl')
T5_PRED    = Path('/Users/xuanhu/Library/CloudStorage/OneDrive-Personal/20260107-20260419/CS4120/dnd_project/part2' + '/models/t5_small/predictions_test.jsonl')

test_raw = [json.loads(l) for l in (DATA_DIR/'test.jsonl').open()]
turn_to_record = {r['turn_id']: r for r in test_raw}
print('Test records:', len(test_raw))

Test records: 216


## 1. Template baseline

Rule-based slot-filling with a handful of handwritten templates per `action_type`. This is the **floor** — it tells us whether our ML models are actually learning something beyond 'fill in the blank'.

In [3]:
def template_generate(r: dict) -> str:
    act = r['action_type']
    actor = r['actor']['name']
    targets = [t['name'] for t in r['targets'] if t.get('name')]
    tgt = targets[0] if targets else 'the foe'
    mech = r['mechanics']
    weapon = mech.get('weapon')
    spell  = mech.get('spell')
    roll   = mech.get('roll') or {}
    damage = mech.get('damage') or []
    dmg_amt = sum(d['amount'] for d in damage) if damage else None
    killed = any(t.get('killed') for t in r['targets'])

    if act == 'attack':
        if roll.get('hit') is False:
            return f"{actor} swings at {tgt} but the blow goes wide."
        if roll.get('crit'):
            return f"{actor} lands a devastating {weapon or 'strike'} on {tgt}!" + (f" {tgt} is slain." if killed else '')
        base = f"{actor} strikes {tgt}" + (f" with a {weapon}" if weapon else '')
        if dmg_amt: base += f" for {dmg_amt} damage"
        base += '.'
        if killed: base += f" {tgt} falls."
        return base
    if act == 'spell':
        base = f"{actor} casts {spell or 'a spell'}"
        if targets: base += f" at {', '.join(targets)}"
        if dmg_amt: base += f", dealing {dmg_amt} damage"
        base += '.'
        return base
    if act == 'heal':
        return f"{actor} channels healing magic, mending wounds."
    if act == 'check':
        return f"{actor} makes the attempt — the result is revealed."
    if act == 'save':
        return f"{actor} braces against the effect."
    return f"{actor} takes a turn."

template_preds = [{'turn_id': r['turn_id'], 'gold': r['narration'],
                   'pred': template_generate(r), 'action_type': r['action_type']}
                  for r in test_raw]
print(template_preds[0])

{'turn_id': 'de61d8dcc41a', 'gold': '“You can be anything you wish!”', 'pred': 'Valentina Cuadrada-Sinesterra casts fly at Emmeline Windsor-Fairfax.', 'action_type': 'spell'}


## 2. Load the other two models' predictions

In [4]:
def load_preds(path):
    return [json.loads(l) for l in Path(path).open()]

ngram_preds = load_preds(NGRAM_PRED) if NGRAM_PRED.exists() else None
t5_preds    = load_preds(T5_PRED)    if T5_PRED.exists()    else None

print('ngram:', 'OK' if ngram_preds else 'MISSING — run 02_train_ngram_lm.ipynb first')
print('t5   :', 'OK' if t5_preds    else 'MISSING — run 03_train_t5_seq2seq.ipynb first')

ngram: OK
t5   : OK


## 3. Metric helpers

In [5]:
rouge = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

def eval_model(preds, label):
    preds = [p for p in preds if p['turn_id'] in turn_to_record]
    gold  = [p['gold'] for p in preds]
    gen   = [p['pred'] for p in preds]

    # BLEU (corpus)
    bleu = sacrebleu.corpus_bleu(gen, [gold]).score

    # ROUGE-L (avg)
    rl = sum(rouge.score(g, p)['rougeL'].fmeasure for g, p in zip(gold, gen)) / max(1, len(gen))

    # Length ratio
    gold_len = sum(len(g.split()) for g in gold) / max(1, len(gold))
    pred_len = sum(len(p.split()) for p in gen)  / max(1, len(gen))

    # Entity faithfulness: does the prediction mention the actor and (if any) a target?
    faith_actor = faith_tgt = both = 0
    n_with_tgt = 0
    for p in preds:
        r = turn_to_record[p['turn_id']]
        actor = r['actor']['name'].lower()
        targets = [t['name'].lower() for t in r['targets'] if t.get('name')]
        blob = p['pred'].lower()
        a_ok = actor and any(tok in blob for tok in actor.split())
        t_ok = any(any(tok in blob for tok in tn.split()) for tn in targets) if targets else None
        faith_actor += int(bool(a_ok))
        if t_ok is not None:
            n_with_tgt += 1
            faith_tgt += int(t_ok)
            both += int(a_ok and t_ok)

    return {
        'model':  label,
        'n':      len(preds),
        'bleu4':  round(bleu, 2),
        'rougeL': round(100*rl, 2),
        'gold_words': round(gold_len, 1),
        'pred_words': round(pred_len, 1),
        'length_ratio': round(pred_len/max(0.1, gold_len), 2),
        'actor_mentioned_pct':  round(100*faith_actor/max(1,len(preds)), 1),
        'target_mentioned_pct': round(100*faith_tgt/max(1,n_with_tgt), 1) if n_with_tgt else None,
        'both_mentioned_pct':   round(100*both/max(1,n_with_tgt), 1)     if n_with_tgt else None,
    }

rows = [eval_model(template_preds, 'Template')]
if ngram_preds: rows.append(eval_model(ngram_preds, 'KN-3 N-gram'))
if t5_preds:    rows.append(eval_model(t5_preds,    'T5-small'))

# Pretty print
cols = list(rows[0].keys())
print('|' + '|'.join(f' {c} ' for c in cols) + '|')
print('|' + '|'.join('---' for _ in cols) + '|')
for row in rows:
    print('|' + '|'.join(f' {row.get(c, "")} ' for c in cols) + '|')

| model | n | bleu4 | rougeL | gold_words | pred_words | length_ratio | actor_mentioned_pct | target_mentioned_pct | both_mentioned_pct |
|---|---|---|---|---|---|---|---|---|---|
| Template | 216 | 0.09 | 6.33 | 22.0 | 9.2 | 0.42 | 100.0 | 95.9 | 95.9 |
| KN-3 N-gram | 216 | 0.22 | 8.63 | 22.0 | 11.2 | 0.51 | 3.2 | 1.5 | 0.0 |
| T5-small | 216 | 0.45 | 10.23 | 22.0 | 11.4 | 0.52 | 56.5 | 29.9 | 16.5 |


## 4. Per-action breakdown (where does each model win/lose?)

In [6]:
def by_action(preds, label):
    out = {}
    groups = defaultdict(list)
    for p in preds:
        groups[p['action_type']].append(p)
    for act, group in groups.items():
        if len(group) < 5: continue
        g = [x['gold'] for x in group]; h = [x['pred'] for x in group]
        bleu = sacrebleu.corpus_bleu(h, [g]).score
        rl = sum(rouge.score(a,b)['rougeL'].fmeasure for a,b in zip(g,h))/len(g)
        out[act] = {'n': len(group), 'bleu': round(bleu,2), 'rougeL': round(100*rl,2)}
    return out

import pandas as pd
combined = {'Template': by_action(template_preds, 'Template')}
if ngram_preds: combined['KN-3'] = by_action(ngram_preds, 'KN-3')
if t5_preds:    combined['T5']   = by_action(t5_preds,    'T5')

# Per-model per-action table
for model, d in combined.items():
    print(f'\n{model}:')
    for act, m in sorted(d.items(), key=lambda x: -x[1]['n']):
        print(f'  {act:8} n={m["n"]:4}  BLEU={m["bleu"]:6}  ROUGE-L={m["rougeL"]:6}')


Template:
  attack   n= 158  BLEU=  0.12  ROUGE-L=  7.29
  spell    n=  50  BLEU=  0.15  ROUGE-L=  3.38
  save     n=   8  BLEU=  0.01  ROUGE-L=  5.88

KN-3:
  attack   n= 158  BLEU=  0.28  ROUGE-L=  8.88
  spell    n=  50  BLEU=  0.18  ROUGE-L=   8.3
  save     n=   8  BLEU=  0.14  ROUGE-L=  5.79

T5:
  attack   n= 158  BLEU=  0.29  ROUGE-L= 10.86
  spell    n=  50  BLEU=  0.24  ROUGE-L=  8.36
  save     n=   8  BLEU=  1.69  ROUGE-L=  9.43


## 5. BERTScore (optional — slow)

Semantic similarity via embedding overlap. Uncomment the cell to run; ~2 minutes on Colab.

In [7]:
# from bert_score import score as bert_score
# 
# def bs(preds):
#     P, R, F1 = bert_score([p['pred'] for p in preds],
#                           [p['gold'] for p in preds],
#                           lang='en', rescale_with_baseline=True, verbose=False)
#     return round(F1.mean().item()*100, 2)
# 
# print('Template:', bs(template_preds))
# if ngram_preds: print('KN-3    :', bs(ngram_preds))
# if t5_preds:    print('T5      :', bs(t5_preds))

## 6. Qualitative side-by-side

The most interesting thing for the report — pick 8 random test records and show all three models alongside the gold.

In [8]:
import random
random.seed(1)
idx = {r['turn_id']: r for r in template_preds}
t5_by_id    = {r['turn_id']: r for r in (t5_preds or [])}
ngram_by_id = {r['turn_id']: r for r in (ngram_preds or [])}

sample_ids = random.sample(list(idx.keys()), k=min(8, len(idx)))
for tid in sample_ids:
    r = turn_to_record[tid]
    print(f'[{r["action_type"]}] actor={r["actor"]["name"]!r}  targets={[t["name"] for t in r["targets"]]}')
    print('  GOLD    :', r['narration'])
    print('  Template:', idx[tid]["pred"])
    if tid in ngram_by_id: print('  KN-3    :', ngram_by_id[tid]["pred"])
    if tid in t5_by_id:    print('  T5      :', t5_by_id[tid]["pred"])
    print()

[attack] actor="Lo'ren Magalia"  targets=['Desnuthga']
  GOLD    : Ren calls upon her healing light again, this time a bit stronger than normal. She summons the healing on Aegis, restoring him to a better state "Now let's finish this one!"
  Template: Lo'ren Magalia strikes Desnuthga with a spirit for 14 damage.
  KN-3    : " You are up to the ground, dead.
  T5      : Lo'ren moves around, hoping to draw his opponent more into the range of some of Ren's most potent spells. But two strikes with his hammer are all it takes to fall them.

[attack] actor='Mick "Ram" Gordon'  targets=['SnakeDoggo1']
  GOLD    : "Come on, how about you give ME a hug?"
  Template: Mick "Ram" Gordon strikes SnakeDoggo1 with a greatsword for 61 damage.
  KN-3    : It seems that Bakari is too smart to fall so quickly. Moving to where she could still fly just fine!
  T5      : Mick slams the snake's greatsword and slams it with a greatsword

[attack] actor='Kraken'  targets=['Merizon Aerialis']
  GOLD    : The Kr

## 7. Report scaffolding — takeaways + limitation

Copy/adapt this into the final report. The rubric rewards **2–3 concrete takeaways + 1 limitation**.

### Expected takeaways (fill in with actual numbers after running)

1. **Specialization vs. fluency trade-off**: Template wins entity faithfulness (~100% — slot-filled) but loses BLEU because it never invents flavor text; T5 loses on entity faithfulness because of decoder hallucination but wins BLEU because it learns FIREBALL idiom.
2. **Where the N-gram breaks**: per-action table — expect `save` / `check` to underperform `attack` (less training data per action-type, rarer templates to memorize).
3. **Length bias**: T5 with beam search tends to over-generate; template under-generates. Look at `length_ratio` in the table.

### Limitation (pick one)

- FIREBALL's gold narrations are **player-authored Discord chat**, not professional narration. ~10–15% of records have mechanical noise (dice notation, emoji, OOC banter) that our cleaning filter doesn't perfectly catch — this caps BLEU/ROUGE upper bounds.
- Our entity-faithfulness metric only checks for name mentions; it doesn't verify that the *relationship* (e.g., who attacked whom) is correctly reproduced.
- With only ~3.5k training records and free-Colab compute, we used `t5-small`. Larger backbones (t5-base, flan-t5-small) would likely close more of the gap to a fine-tuned LLM baseline.

In [9]:
# Save comparison table for the report
import pandas as pd
df = pd.DataFrame(rows)
df.to_csv('/Users/xuanhu/Library/CloudStorage/OneDrive-Personal/20260107-20260419/CS4120/dnd_project/part2' + '/results_comparison.csv', index=False)
df.to_markdown('/Users/xuanhu/Library/CloudStorage/OneDrive-Personal/20260107-20260419/CS4120/dnd_project/part2' + '/results_comparison.md', index=False)
df

,model,n,bleu4,rougeL,gold_words,pred_words,length_ratio,actor_mentioned_pct,target_mentioned_pct,both_mentioned_pct
0,Template,216,0.09,6.33,22.0,9.2,0.42,100.0,95.9,95.9
1,KN-3 N-gram,216,0.22,8.63,22.0,11.2,0.51,3.2,1.5,0.0
2,T5-small,216,0.45,10.23,22.0,11.4,0.52,56.5,29.9,16.5
